# 14 Prepare CRSP stock monthly and daily

目标：清洗 CRSP 月度和日度股票数据，并按锁定的 CIZ 字段建立统一普通股基础宇宙。

- 输入：monthly_stock_file.csv、daily_stock_file.csv、Fama–French 月/日因子
- 正式输出：stock_monthly_clean.parquet、stock_daily_clean.parquet、因子表和宇宙审计
- 主键：permno+month；permno+date
- 上游依赖：01
- smoke 模式仅使用真实本地数据子样本，并写入 `data/audit/smoke/`；不冒充正式结果。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic, write_parquet_atomic
from src.notebook14_determinism import build_monthly_crsp_deterministic
from src.runtime_paths import output_path

daily_columns=["PERMNO","DlyCalDt","DlyRet","DlyPrc","DlyVol","DlyBid","DlyAsk"]
def esc(path): return str(path).replace("'","''")
limit_day="" if not SMOKE else " LIMIT 1000000"
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET preserve_insertion_order=false")

def read_ff(path,daily=False):
    x=pd.read_csv(path,dtype=str); x=x.rename(columns={x.columns[0]:"factor_date"})
    names=[c for c in x.columns if c!="factor_date"]
    for c in names: x[c]=pd.to_numeric(x[c].str.strip(),errors="coerce")/100.0
    x["date"]=pd.to_datetime(x["factor_date"],format="%Y%m%d" if daily else "%Y%m",errors="coerce")
    if not daily: x["date"]=x["date"].dt.to_period("M").dt.to_timestamp()
    return x.dropna(subset=["date"])[["date"]+names]
ffm=read_ff(PATHS["ff_monthly"]).merge(read_ff(PATHS["mom_monthly"]),on="date",how="left",validate="one_to_one")
ffd=read_ff(PATHS["ff_daily"],daily=True)

out1=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_monthly_clean.parquet")
out2=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_daily_clean.parquet")
out3=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/ff_monthly_clean.parquet")
out4=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/ff_daily_clean.parquet")
audit=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/audit/14_common_stock_universe.csv")
conflict_audit=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/audit/notebook14_conflict_audit.csv")
exact_duplicate_audit=output_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/audit/14_exact_duplicate_audit.csv")
staged_monthly=PATHS["full_run_tmp"]/"14_stock_monthly_clean_staged.parquet"
normalized_monthly=PATHS["full_run_tmp"]/"14_stock_monthly_normalized_staged.parquet"
staged_daily=PATHS["full_run_tmp"]/"14_stock_daily_clean_staged.parquet"
staged_monthly.unlink(missing_ok=True); normalized_monthly.unlink(missing_ok=True)
staged_daily.unlink(missing_ok=True)
monthly_stats=build_monthly_crsp_deterministic(
    con,
    source_path=PATHS["crsp_monthly"],
    normalized_stage_path=normalized_monthly,
    clean_stage_path=staged_monthly,
    conflict_audit_path=conflict_audit,
    exact_duplicate_audit_path=exact_duplicate_audit,
    limit_rows=500000 if SMOKE else None,
)
daily_projection=",".join(daily_columns)
daily_query=f"""WITH raw AS (
 SELECT {daily_projection}
 FROM read_csv_auto('{esc(PATHS["crsp_daily"])}',header=true,all_varchar=true)
 {limit_day}
), casted AS (
 SELECT
   try_cast(PERMNO AS BIGINT) permno,DlyCalDt dlycaldt,
   try_cast(DlyCalDt AS DATE) date_value,
   CASE WHEN try_cast(DlyRet AS DOUBLE) IN (-66,-77,-88,-99) THEN NULL
        ELSE try_cast(DlyRet AS DOUBLE) END dlyret,
   try_cast(DlyPrc AS DOUBLE) dlyprc,try_cast(DlyVol AS DOUBLE) dlyvol,
   try_cast(DlyBid AS DOUBLE) dlybid,try_cast(DlyAsk AS DOUBLE) dlyask
 FROM raw
), ranked AS (
 SELECT *,row_number() OVER(PARTITION BY permno,date_value
   ORDER BY dlycaldt DESC,dlyprc DESC,dlyvol DESC) duplicate_rank
 FROM casted WHERE permno IS NOT NULL AND date_value IS NOT NULL
)
SELECT * EXCLUDE(date_value,duplicate_rank),date_value AS "date"
FROM ranked WHERE duplicate_rank=1"""
duckdb_copy_parquet_atomic(con,daily_query,staged_daily)
con.execute(f"CREATE TEMP VIEW monthly_clean AS SELECT * FROM read_parquet('{esc(staged_monthly)}')")
con.execute(f"CREATE TEMP VIEW daily_clean AS SELECT * FROM read_parquet('{esc(staged_daily)}')")
duckdb_copy_csv_atomic(con,"""SELECT "month",
 count(*) n_all,
 count(*) FILTER(WHERE common_stock) n_common_stock,
 count(*) FILTER(WHERE sharetype<>'NS' OR sharetype IS NULL) excluded_sharetype,
 count(*) FILTER(WHERE securitytype<>'EQTY' OR securitytype IS NULL) excluded_securitytype,
 count(*) FILTER(WHERE securitysubtype<>'COM' OR securitysubtype IS NULL)
   excluded_securitysubtype
 FROM monthly_clean GROUP BY "month" ORDER BY "month" """,audit)
monthly_clean_count=int(con.execute("SELECT count(*) FROM monthly_clean").fetchone()[0])
daily_clean_count=int(con.execute("SELECT count(*) FROM daily_clean").fetchone()[0])
common_count=int(con.execute("SELECT count(*) FROM monthly_clean WHERE common_stock").fetchone()[0])
monthly_dupes=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM monthly_clean GROUP BY permno,"month" HAVING count(*)>1)""").fetchone()[0])
daily_dupes=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM daily_clean GROUP BY permno,"date" HAVING count(*)>1)""").fetchone()[0])
assert monthly_dupes==0 and daily_dupes==0
con.close()
write_parquet_atomic(ffm,out3); write_parquet_atomic(ffd,out4)
out1.parent.mkdir(parents=True,exist_ok=True)
os.replace(staged_monthly,out1); os.replace(staged_daily,out2)
normalized_monthly.unlink(missing_ok=True)
raw_monthly_count=monthly_stats["raw_record_count"]
raw_daily_count=min(110285669,1000000) if SMOKE else 110285669
INPUTS_USED=[p.name for p in [PATHS["crsp_monthly"],PATHS["crsp_daily"],PATHS["ff_monthly"],PATHS["mom_monthly"],PATHS["ff_daily"]]]
OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out1,out2,out3,out4,audit,conflict_audit,exact_duplicate_audit]]
INPUT_COUNTS={"monthly":raw_monthly_count,"daily":raw_daily_count}; OUTPUT_COUNTS={"monthly_clean":monthly_clean_count,"daily_clean":daily_clean_count,"common_stock":common_count,"monthly_exact_duplicate_records_removed":monthly_stats["exact_duplicate_record_count"],"monthly_conflict_keys_excluded":monthly_stats["conflict_key_count"],"monthly_conflict_records_excluded":monthly_stats["conflict_record_count"],"monthly_conflict_keys_affecting_common_stock":monthly_stats["affected_common_stock_key_count"]}
PRIMARY_KEY=["permno","month"]; DUPLICATE_KEY_COUNT=monthly_dupes+daily_dupes; EXCLUDED_ROW_COUNT=(raw_monthly_count-monthly_clean_count)+(raw_daily_count-daily_clean_count); EXCEPTION_FILE=str(conflict_audit.relative_to(ROOT))
STAGE_STATUS="passed"; STAGE_NOTES=["Exact duplicate monthly records are audited and collapsed by their complete normalized content.","Every PERMNO-month with more than one distinct normalized content record is wholly excluded and all raw occurrences are retained in notebook14_conflict_audit.csv.","No first/last row, CUSIP, common-stock status or input position is used to resolve a monthly conflict.","FF factors converted from percent to decimal exactly once.","The same locked CIZ common-stock flag is persisted for ranking, PCA and direction estimation.","The 56.22GB daily CSV was projected to seven required columns and processed by external-memory DuckDB without Pandas materialization."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "14_prepare_crsp_stock_monthly_daily", stage="14", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
